# Main for Data Analysis 

Filters analyses and plots track and tour data

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and before main_energy_sim.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import data_handling.data_processing as dp

from utils.style_config import *

In [ ]:
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'sans-serif',
    'font.sans-serif': 'Arial',
    'font.size': 12,
    'text.latex.preamble': r'\usepackage{siunitx}'
})

## Preprocessing 

### Data filtering
Load and preprocess the raw trips data.

Preprocessing includes, filtering out tracks shorter than 1 km, with avg. speeds > max. speed and with negative durations. Additioally adds forwarder data to each trip.

In [ ]:
# Define LOCATIONS
LOCATIONS = dp.LOCATIONS

# Load data
df_trips_unfiltered, df_fleet= dp.load_data()
# Preprocess data
df_trips = dp.preprocess_trips_data(df_trips_unfiltered, df_fleet)

# Fixes tours that 'spill over' into the next tour and cause tracks to be assigned to the wrong track
# This is a problem in the raw data (df_trips_unfiltered)
df_trips = dp.fix_faulty_tour_assignment(df_trips, df_trips_unfiltered)

# Fixes assigns a locations that corresponds to a home base to the end of each tour, if it does not
# do so anyway. Usually because the last track of a tour has been filtered out in preprocess_trips_data()
# If new home bases are addded, update the homebase_map in fix_faulty_tour_endings()
df_trips = dp.fix_faulty_tour_endings(df_trips, df_trips_unfiltered)
#df_trips.to_csv('./input/stations/tracks_filtered.csv')

### Process truck occupations and resample dataframes

In [ ]:
df_stops = dp.process_stops_data(df_trips)

# Cumulative time spent driving or in each type of location for each truck
df_rt_joined_plot = dp.calculate_rest_times_and_driving(df_stops, df_trips)

# Resample occupation data
# shows time spent (in min) not occurencess
truck_day_occ = dp.resample_occupation_data(df_stops, df_trips)

### Create tour statistics 

Aggregate all tracks that belong to a single tour into a row 

Creates: input/stations/tours.csv

In [ ]:
df_tours = dp.aggregate_tours(df_trips, soc_min=0.15, save=True)

# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000
distances = [300, 375, 500, 1000]
percentages = {dist: (df_tours[df_tours['distance_km'] < dist].shape[0] / df_tours.shape[0]) * 100 for dist in distances}

for dist, perc in percentages.items():
    print(f"Tours under {dist} km: {perc:.2f}%")

print('\n')  
# Calculate the percentage of tours with distance_km < 300, < 375, < 500, < 1000 per freight forwarder
ff_percentages = {}
for ff in df_tours['freight_forwarder'].unique():
    ff_data = df_tours[df_tours['freight_forwarder'] == ff]
    ff_percentages[ff] = {dist: (ff_data[ff_data['distance_km'] < dist].shape[0] / ff_data.shape[0]) * 100 for dist in distances}

# Print the results
for ff, stats in ff_percentages.items():
    print(f"Freight Forwarder {ff}:")
    for dist, perc in stats.items():
        print(f"  Tours under {dist} km: {perc:.2f}%")

### Meta data 

In [ ]:
# Calculation of general meta data
meta_data = dp.calculate_meta_data(df_trips_unfiltered, df_trips, df_fleet)
# Converting meta data to DataFrame
general_df, _, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(general_df)

# Converting meta data to DataFrame
_, temporal_df, _, _ = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(temporal_df)

# Converting meta data to DataFrame
_, _, _, ff_df = dp.meta_data_to_df(meta_data)

# Display of DataFrame
display(ff_df)

## Plots

### Data Recordings - Weekly Distance    

In [ ]:
def plot_weekly_distances(df_trips): #TODO: rename to plot_weekly_distances and delete old version in code
    """
    Plot the weekly distances covered by the fleet as a stacked bar plot, centered on week start dates (Sunday).
    """
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(textwidth, h_169))

    # Preprocess
    df_trips['start_time_7d'] = pd.to_datetime(df_trips['start_time'], utc=True).dt.to_period("W-SUN")
    distances_km = df_trips.groupby(['start_time_7d', 'freight_forwarder'])['distance'].sum().unstack(fill_value=0) / 1000

    # Get first and last recording date for each fleet
    fleet_periods = df_trips.groupby('freight_forwarder').agg(
        first_date=('start_time', 'min'),
        last_date=('start_time', 'max')
    )

    # Create a color mapping for freight forwarders
    freight_forwarders = sorted(df_trips['freight_forwarder'].unique())
    colors_plot = ['#1f77b4', '#2c2c2c', '#ff7f0e', '#a0c114', '#7F7F7F', '#B3679B']
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]

    # Plot
    distances_km.plot.bar(stacked=True, xlabel='', ylabel='distance / km', ax=ax, color=[ff_colors.get(i, '#808080') for i in distances_km.columns])

    # Configure the plot
    ax.set_ylim(0, 82000)  # Capped at 85,000 for y-axis
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10000))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(5000))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda t, pos: f"{int(t):,}"))
    ax.set_ylabel('distance / km', fontsize=14)
    ax.set_xlabel('week', fontsize=14)

    # X-axis ticks (show all weeks for readability with larger font)
    x_week_ticks = range(1,75,3)
    ax.xaxis.set_minor_locator(ticker.FixedLocator(x_week_ticks))
    ax.xaxis.set_minor_formatter(ticker.FuncFormatter(lambda t, pos: f"{distances_km.index[t].start_time.day:02d}-{distances_km.index[t].start_time.month:02d}-{distances_km.index[t].start_time.year}"))
    
    # Remove major year ticks to avoid clutter, since we add dividers
    ax.xaxis.set_major_locator(ticker.NullLocator())

    ax.tick_params(axis="y", labelsize=13)
    ax.tick_params(axis="x", which="minor", pad=20, labelrotation=90, labelsize=11)  
    ax.grid(axis="y", which='major', alpha=0.3)
    ax.grid(axis="y", which='minor', alpha=0.15)

    # Add horizontal lines for fleet recording periods
    indicator_level = 70000
    max_fleet = max(fleet_periods.index) if not fleet_periods.empty else 6  
    for i, (fleet, (first_date, last_date)) in enumerate(fleet_periods.iterrows()):
        first_week = pd.Timestamp(first_date).to_period("W-SUN").start_time
        last_week = pd.Timestamp(last_date).to_period("W-SUN").start_time
        first_idx = np.searchsorted(distances_km.index.to_timestamp(), first_week)
        last_idx = np.searchsorted(distances_km.index.to_timestamp(), last_week)
        ax.plot([first_idx, last_idx], [indicator_level + i * 2000, indicator_level + i * 2000], 
                '--|', linewidth=1, markersize=7, color=ff_colors.get(fleet, 'gray'))

    # Create year divider for 2021-2022
    year_change_idx_2021_2022 = np.searchsorted(distances_km.index.to_timestamp(), pd.Timestamp('2022-01-01').to_period("W-SUN").start_time)
    ax.axvline(x=year_change_idx_2021_2022 - 0.5, color='black', linestyle='--', alpha=0.5)

    # Create year divider for 2022-2023
    year_change_idx_2022_2023 = np.searchsorted(distances_km.index.to_timestamp(), pd.Timestamp('2023-01-01').to_period("W-SUN").start_time)
    ax.axvline(x=year_change_idx_2022_2023 - 0.5, color='black', linestyle='--', alpha=0.5)

    # Add legend (moved to upper right, entries listed vertically, kept within plot)
    ax.legend(ncol=1, title="fleet", loc='upper right', bbox_to_anchor=(1.0,0.9),fontsize=13, title_fontsize=14)

    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25, right=0.80)  

    # Save the plot
    plt.savefig('output/figures/operational/data_recordings.pdf', bbox_inches='tight')
    plt.show()


plot_weekly_distances(df_trips)  

### KDE Plots - Tracks - Distance | Duration | Max Speed | Avg Speed

In [ ]:

# -------------------------------------------------------------------------------------------
#              KDE PLOTS - TRACK - DISTANCE | DURATION | MAX SPEED | AVG SPEED
# -------------------------------------------------------------------------------------------


def plot_kde_plots(df_trips):
    """
    Plot four KDE plots in a 2x2 grid to show the distribution of the data with vehicle IDs coloring.
    Vehicles from the same freight forwarder will share the same color.
    """
    textwidth = 159.2 / 25.4  # Umrechnung der Textbreite von mm in Zoll
    height = textwidth * 1.2  # Etwas höhere Höhe
    width = textwidth * 1.5  # Größere Breite
    
    fig, axes = plt.subplots(2, 2, figsize=(width, height), sharey=False)

    # Group vehicle IDs by freight forwarder
    vehicle_to_ff = df_trips[['vehicle_id', 'freight_forwarder']].drop_duplicates().set_index('vehicle_id')['freight_forwarder']
    freight_forwarders = sorted(df_trips['freight_forwarder'].unique())
    
    # Create a color mapping for freight forwarders
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]  # Loop through the colors if there are more FFs than colors
    
    # Create palette that assigns the same color to all vehicles from the same freight forwarder
    palette = {vehicle_id: ff_colors[vehicle_to_ff.loc[vehicle_id]] for vehicle_id in df_trips['vehicle_id'].unique()}

    # Create a mapping for the colorbar - we'll map vehicle IDs to their FF colors but maintain numeric ordering
    n_vehicles = df_trips['vehicle_id'].nunique()
    norm = Normalize(vmin=1, vmax=n_vehicles)
    vehicle_ids = sorted(df_trips['vehicle_id'].unique())
    cmap_colors = [ff_colors[vehicle_to_ff.loc[vid]] for vid in vehicle_ids]
    custom_cmap = ListedColormap(cmap_colors)

    sns.kdeplot(data=df_trips, x='distance_km', hue='vehicle_id', ax=axes[0, 0], palette=palette, common_norm=False, fill=True)
    axes[0, 0].set_xlabel('Distance / km')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].legend_.remove()
    axes[0, 0].set_xlim(0, 125)
    mean_distance = df_trips['distance_km'].mean()
    std_distance = df_trips['distance_km'].std()
    median_distance = df_trips['distance_km'].median()
    axes[0, 0].axvline(mean_distance, color='r', linestyle='--')
    axes[0, 0].axvline(mean_distance + std_distance, color='r', linestyle=':')
    axes[0, 0].axvline(mean_distance - std_distance, color='r', linestyle=':')
    axes[0, 0].axvline(median_distance, color='orange', linestyle='-')
    # Add labels in top right corner
    axes[0, 0].text(0.95, 0.95, f'Mean: {mean_distance:.1f} km\nMedian: {median_distance:.1f} km', 
                   transform=axes[0, 0].transAxes, ha='right', va='top',
                   fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    # Add grid
    axes[0, 0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    sns.kdeplot(data=df_trips, x='duration_h', hue='vehicle_id', ax=axes[0, 1], palette=palette, common_norm=False, fill=True)
    axes[0, 1].set_xlabel('Duration / h')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].legend_.remove()
    axes[0, 1].set_xlim(0, 4)
    mean_duration = df_trips['duration_h'].mean()
    std_duration = df_trips['duration_h'].std()
    median_duration = df_trips['duration_h'].median()
    axes[0, 1].axvline(mean_duration, color='r', linestyle='--')
    axes[0, 1].axvline(mean_duration + std_duration, color='r', linestyle=':')
    axes[0, 1].axvline(mean_duration - std_duration, color='r', linestyle=':')
    axes[0, 1].axvline(median_duration, color='orange', linestyle='-')
    # Add labels in top right corner
    axes[0, 1].text(0.95, 0.95, f'Mean: {mean_duration:.2f} h\nMedian: {median_duration:.2f} h', 
                   transform=axes[0, 1].transAxes, ha='right', va='top',
                   fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    # Add grid
    axes[0, 1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    sns.kdeplot(data=df_trips, x='max_speed_kmh', hue='vehicle_id', ax=axes[1, 0], palette=palette, common_norm=False, fill=True)
    axes[1, 0].set_xlabel('Max Speed / km/h')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].legend_.remove()
    axes[1, 0].set_xlim(0, 150)
    mean_max_speed = df_trips['max_speed_kmh'].mean()
    std_max_speed = df_trips['max_speed_kmh'].std()
    median_max_speed = df_trips['max_speed_kmh'].median()
    axes[1, 0].axvline(mean_max_speed, color='r', linestyle='--')
    axes[1, 0].axvline(mean_max_speed + std_max_speed, color='r', linestyle=':')
    axes[1, 0].axvline(mean_max_speed - std_max_speed, color='r', linestyle=':')
    axes[1, 0].axvline(median_max_speed, color='orange', linestyle='-')
    # Add labels in top right corner
    axes[1, 0].text(0.95, 0.95, f'Mean: {mean_max_speed:.1f} km/h\nMedian: {median_max_speed:.1f} km/h', 
                   transform=axes[1, 0].transAxes, ha='right', va='top',
                   fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    # Add grid
    axes[1, 0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    sns.kdeplot(data=df_trips, x='avg_speed_kmh', hue='vehicle_id', ax=axes[1, 1], palette=palette, common_norm=False, fill=True)
    axes[1, 1].set_xlabel('Average Speed / km/h')
    axes[1, 1].set_ylabel('Density')
    axes[1, 1].legend_.remove()
    axes[1, 1].set_xlim(0, 100)
    mean_avg_speed = df_trips['avg_speed_kmh'].mean()
    std_avg_speed = df_trips['avg_speed_kmh'].std()
    median_avg_speed = df_trips['avg_speed_kmh'].median()
    axes[1, 1].axvline(mean_avg_speed, color='r', linestyle='--')
    axes[1, 1].axvline(mean_avg_speed + std_avg_speed, color='r', linestyle=':')
    axes[1, 1].axvline(mean_avg_speed - std_avg_speed, color='r', linestyle=':')
    axes[1, 1].axvline(median_avg_speed, color='orange', linestyle='-')
    # Add labels in top right corner
    axes[1, 1].text(0.95, 0.95, f'Mean: {mean_avg_speed:.1f} km/h\nMedian: {median_avg_speed:.1f} km/h', 
                   transform=axes[1, 1].transAxes, ha='right', va='top',
                   fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    # Add grid
    axes[1, 1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

    # Adjust layout to make room for the single legend on the right
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    
    # Add a single color bar on the right showing vehicle IDs but colored by their freight forwarder
    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.02, pad=0.04, aspect=30, shrink=0.8)
    cbar.ax.invert_yaxis()
    
    # Create a legend for the lines (mean, std dev, median) that appears above the colorbar
    red_lines = [plt.Line2D([0], [0], color='r', linestyle='--', label='Mean'),
                 plt.Line2D([0], [0], color='r', linestyle=':', label='1 Std. Dev.'),
                 plt.Line2D([0], [0], color='orange', linestyle='-', label='Median')]
    
    # Add the legend directly above the colorbar
    fig.legend(handles=red_lines, loc='upper right', bbox_to_anchor=(0.95, 0.99), 
              fontsize=9, frameon=True, title="Statistics")
    
    # Add freight forwarder legend to explain the colors
    handles = [mpatches.Patch(color=color, label=f'FF {ff}') for ff, color in ff_colors.items()]
    fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(0.95, 0.222), 
              fontsize=9, frameon=True, title="Freight Forwarders")
    
    cbar.set_label('Vehicle IDs', labelpad=15, fontsize=12)
    
    # plt.savefig('output/figures/operational/kde_plots.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/kde_plots.pdf', bbox_inches='tight')
        
    plot_kde_plots(df_trips)

### KDE Plots - Tours - Distance | Duration | Max Speed | Avg Speed

In [ ]:

def plot_tour_kde_plots(df_trips):
    """
    Plot two KDE plots in a 1x2 grid to show the distribution of cumulative distances and durations by tour.
    Tours are colored by freight forwarder.
    
    Parameters:
    -----------
    df_trips : pandas DataFrame
        DataFrame containing trip data with tour_id, vehicle_id, freight_forwarder, distance_km, and duration_h columns
    """
    # First, calculate cumulative values by tour_id
    tour_aggregates = df_trips.groupby(['tour_id', 'freight_forwarder', 'vehicle_id']).agg({
        'distance_km': 'sum',  # Sum of all distances in the tour
        'duration_h': 'sum'    # Sum of all durations in the tour
    }).reset_index()
    
    # Set figure dimensions
    textwidth = 159.2 / 25.4  # Convert text width from mm to inches
    height = textwidth * 0.8   # Increase height to accommodate legends at the top
    
    # Create figure with more space at the top for legends
    fig = plt.figure(figsize=(textwidth, height))
    
    # Create gridspec for more control over subplot placement
    gs = fig.add_gridspec(2, 2, height_ratios=[0.2, 0.8])
    
    # Create a row for legends at the top
    legend_ax = fig.add_subplot(gs[0, :])
    legend_ax.axis('off')  # Hide axes for the legend area
    
    # Create the two plot axes in the second row
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[1, 1])
    axes = [ax1, ax2]
    
    # Create a color mapping for freight forwarders
    freight_forwarders = sorted(tour_aggregates['freight_forwarder'].unique())
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]
    
    # Create palette that assigns colors based on freight forwarder
    palette = {vehicle_id: ff_colors[ff] for vehicle_id, ff in 
              tour_aggregates[['vehicle_id', 'freight_forwarder']].drop_duplicates().values}
    
    # Plot distance KDE
    sns.kdeplot(data=tour_aggregates, x='distance_km', hue='vehicle_id', 
                ax=axes[0], palette=palette, common_norm=False, fill=True)
    axes[0].set_xlabel('Cumulative Tour Distance / km')
    axes[0].set_ylabel('Density')
    axes[0].legend_.remove()
    
    # Calculate and show statistics for distance
    mean_distance = tour_aggregates['distance_km'].mean()
    std_distance = tour_aggregates['distance_km'].std()
    median_distance = tour_aggregates['distance_km'].median()
    
    # Set reasonable x-limit based on data
    max_xlim_distance = min(800, tour_aggregates['distance_km'].quantile(0.99))
    axes[0].set_xlim(0, max_xlim_distance)
    
    # Add reference lines
    axes[0].axvline(mean_distance, color='r', linestyle='--')
    axes[0].axvline(mean_distance + std_distance, color='r', linestyle=':')
    axes[0].axvline(mean_distance - std_distance, color='r', linestyle=':')
    axes[0].axvline(median_distance, color='orange', linestyle='-')
    
    # Add statistics box
    axes[0].text(0.95, 0.95, f'Mean: {mean_distance:.1f} km\nMedian: {median_distance:.1f} km', 
               transform=axes[0].transAxes, ha='right', va='top',
               fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    
    # Add grid
    axes[0].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    
    # Plot duration KDE
    sns.kdeplot(data=tour_aggregates, x='duration_h', hue='vehicle_id', 
                ax=axes[1], palette=palette, common_norm=False, fill=True)
    axes[1].set_xlabel('Cumulative Tour Duration / h')
    axes[1].set_ylabel('Density')
    axes[1].legend_.remove()
    
    # Calculate and show statistics for duration
    mean_duration = tour_aggregates['duration_h'].mean()
    std_duration = tour_aggregates['duration_h'].std()
    median_duration = tour_aggregates['duration_h'].median()
    
    # Set reasonable x-limit based on data
    max_xlim_duration = min(30, tour_aggregates['duration_h'].quantile(0.99))
    axes[1].set_xlim(0, max_xlim_duration)
    
    # Add reference lines
    axes[1].axvline(mean_duration, color='r', linestyle='--')
    axes[1].axvline(mean_duration + std_duration, color='r', linestyle=':')
    axes[1].axvline(mean_duration - std_duration, color='r', linestyle=':')
    axes[1].axvline(median_duration, color='orange', linestyle='-')
    
    # Add statistics box
    axes[1].text(0.95, 0.95, f'Mean: {mean_duration:.2f} h\nMedian: {median_duration:.2f} h', 
               transform=axes[1].transAxes, ha='right', va='top',
               fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    
    # Add grid
    axes[1].grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
    
    # Create a legend for the statistics
    red_lines = [plt.Line2D([0], [0], color='r', linestyle='--', label='Mean'),
                 plt.Line2D([0], [0], color='r', linestyle=':', label='1 Std. Dev.'),
                 plt.Line2D([0], [0], color='orange', linestyle='-', label='Median')]
    
    # Create freight forwarder legend handles
    ff_handles = [mpatches.Patch(color=color, label=f'FF {ff}') for ff, color in ff_colors.items()]
    
    # Add legends to the top area
    # First legend: Statistics (placed on the left side)
    stats_legend = legend_ax.legend(handles=red_lines, loc='upper left', 
                                    bbox_to_anchor=(0.01, 0.9),
                                    fontsize=9, frameon=True, title="Statistics")
    legend_ax.add_artist(stats_legend)  # Add the first legend
    
    # Second legend: Freight forwarders (placed on the right side)
    # Adjust ncol to display in 3x2 grid (3 columns, 2 rows if there are 6 freight forwarders)
    ncols = min(3, len(ff_colors))
    ff_legend = legend_ax.legend(handles=ff_handles, loc='upper right', 
                                bbox_to_anchor=(0.99, 0.9),
                                fontsize=9, frameon=True, title="Freight Forwarders",
                                ncol=ncols)  # 3 columns
    
    plt.tight_layout()
    # plt.savefig('output/figures/operational/tour_kde_plots.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/tour_kde_plots.pdf', bbox_inches='tight')

    
plot_tour_kde_plots(df_trips)

### Weekly Distance Boxplots    

In [ ]:


def plot_weekly_distance_boxplot(df_trips):
    """
    Creates a figure with a central boxplot showing all freight forwarders' data together
    at the top, and individual plots for each freight forwarder below.
    
    Parameters:
    -----------
    df_trips : DataFrame
        DataFrame containing trip data with vehicle_id, start_time, distance_km, and freight_forwarder columns
    """
    df_trips['start_time'] = pd.to_datetime(df_trips['start_time'], utc=True)
    df_trips['weekday'] = df_trips['start_time'].dt.weekday  # 0 = Monday, 6 = Sunday
    df_trips['date'] = df_trips['start_time'].dt.date
    daily_distance = df_trips.groupby(['vehicle_id', 'date', 'weekday', 'freight_forwarder'])['distance_km'].sum().reset_index()

    # Define plot size
    textwidth = (159.2 / 25.4)*2  # Convert text width from mm to inches
    h_169 = 9 / 16 * textwidth
    height = h_169 * 2  # Increase height for better readability
    width = textwidth * 1.2  # Increase width for better layout

    # Define colors for weekdays and average
    weekday_colors = [
        colors['TUMBlue4'], colors['TUMBlue3'], colors['TUMBlue2'], 
        colors['TUMBlue1'], colors['TUMGray3'], colors['TUMGray2'], 
        colors['TUMGray1']
    ]
    avg_color = colors['LightPurple']  # Special color for the average column

    # Create subplots for each freight forwarder plus one for the central plot
    fig = plt.figure(figsize=(width, height))
    
    # Add a GridSpec with 4 rows and 2 columns
    gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1])
    
    # Add the central plot at the top, spanning both columns
    central_ax = fig.add_subplot(gs[0, :])
    
    # Create the individual freight forwarder axes
    axes = []
    for i in range(1, 4):  # Rows 1-3
        for j in range(2):  # 2 columns
            ax = fig.add_subplot(gs[i, j])
            axes.append(ax)
    
    # Get unique freight forwarders
    freight_forwarders = sorted(df_trips['freight_forwarder'].unique())
    
    # Prepare data for the central plot
    all_boxplot_data = []
    for weekday in range(7):
        weekday_data = daily_distance[daily_distance['weekday'] == weekday]['distance_km'].tolist()
        all_boxplot_data.append(weekday_data)
    
    # Add the average data for the central plot (all weekdays combined)
    all_avg_data = daily_distance['distance_km'].tolist()
    all_boxplot_data.append(all_avg_data)
    
    # Create central boxplot with all data
    scaling_factor = textwidth / 13
    bp_central = central_ax.boxplot(
        all_boxplot_data,
        patch_artist=True,
        showmeans=True,
        meanline=True,
        meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
        medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
        showfliers=True,
        flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
        whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        widths=0.55 * scaling_factor,
        positions=range(len(all_boxplot_data)),
        labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Combined']
    )
    
    # Style central boxplot
    central_ax.set_title('All Freight Forwarders Combined', fontsize=14 * scaling_factor)
    central_ax.tick_params(axis='x', labelsize=12 * scaling_factor)
    central_ax.tick_params(axis='y', labelsize=12 * scaling_factor)
    central_ax.grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
    central_ax.set_ylim(bottom=0)
    
    # Assign colors to the central boxplots (7 weekdays + average)
    for i, (box, color) in enumerate(zip(bp_central['boxes'], weekday_colors + [avg_color])):
        box.set_facecolor(color)
        # Add a slightly thicker border to the average box to make it stand out
        if i == 7:  # Average box
            box.set(linewidth=2.0 * scaling_factor)
    
    # Annotate central plot median values - moved to the left to overlap with the median line
    for n, (median_feature, distances) in enumerate(zip(bp_central['medians'], all_boxplot_data)):
        distances_series = pd.Series(distances)
        median_value = distances_series.median()
        x_median, y_median = median_feature.get_xydata()[1]
        # Move text position more to the left (was +0.25, now 0.0 to center on the median line)
        central_ax.text(x_median, y_median, f'{median_value:.2f}', 
                horizontalalignment='center', color=colors['TUMBlack'], fontsize=10 * scaling_factor, 
                bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                        edgecolor=colors['TUMOrange'], alpha=0.9))
    
    # Process each freight forwarder for individual plots
    for i, ff in enumerate(freight_forwarders):
        if i >= len(axes):
            break
            
        # Filter data for current freight forwarder
        ff_data = daily_distance[daily_distance['freight_forwarder'] == ff]
        
        # Calculate and print daily distance per truck for the current freight forwarder
        print(f"\nFreight Forwarder {ff} - Daily Distance per Truck:")
        weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        daily_stats = ff_data.groupby('weekday')['distance_km'].agg(['mean', 'std', 'median', 'count']).reset_index()
        daily_stats['weekday_name'] = daily_stats['weekday'].apply(lambda x: weekday_names[x])
        print(daily_stats[['weekday_name', 'mean', 'std', 'median', 'count']])
        print(f"Overall mean: {ff_data['distance_km'].mean():.2f} km")
                
        # Group by weekday and prepare boxplot data
        boxplot_data = ff_data.groupby('weekday')['distance_km'].apply(list)
        
        # Add the average data for this freight forwarder (all weekdays combined)
        avg_data = ff_data['distance_km'].tolist()
        boxplot_data_list = list(boxplot_data) + [avg_data]
        
        # Create boxplot
        bp = axes[i].boxplot(
            boxplot_data_list,
            patch_artist=True,
            showmeans=True,
            meanline=True,
            meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
            medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
            showfliers=True,
            flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
            whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            widths=0.55 * scaling_factor,
            positions=range(len(boxplot_data_list)),
            labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Combined']
        )
        
        # Set x-axis and y-axis tick labels font size
        axes[i].tick_params(axis='x', labelsize=12 * scaling_factor)
        axes[i].tick_params(axis='y', labelsize=12 * scaling_factor)
        
        # Annotate median values - moved to the left to overlap with the median line
        for n, (median_feature, distances) in enumerate(zip(bp['medians'], boxplot_data_list)):
            distances_series = pd.Series(distances)
            median_value = distances_series.median()
            x_median, y_median = median_feature.get_xydata()[1]
            # Move text position more to the left (was +0.25, now 0.0 to center on the median line)
            axes[i].text(x_median, y_median, f'{median_value:.2f}', 
                    horizontalalignment='center', color=colors['TUMBlack'], fontsize=10 * scaling_factor, 
                    bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                            edgecolor=colors['TUMOrange'], alpha=0.9))
        
        # Assign colors to the boxplots
        for j, (box, color) in enumerate(zip(bp['boxes'], weekday_colors + [avg_color])):
            box.set_facecolor(color)
            # Add a slightly thicker border to the average box to make it stand out
            if j == 7:  # Average box
                box.set(linewidth=2.0 * scaling_factor)
        
        # Set title for each subplot
        axes[i].set_title(f'Freight Forwarder {ff}', fontsize=12 * scaling_factor)
        
        # Add grid
        axes[i].grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
        
        # Set y-axis limit
        axes[i].set_ylim(bottom=0)
        
        # Customize spines
        for spine in axes[i].spines.values():
            spine.set_edgecolor(colors['TUMBlack'])
            spine.set_linewidth(0.8 * scaling_factor)
    
    # Remove any empty subplots if there are fewer than 6 freight forwarders
    for i in range(len(freight_forwarders), len(axes)):
        fig.delaxes(axes[i])
    
    # Add common labels
    fig.text(0.5, 0.01, 'Weekday', ha='center', va='center', fontsize=16 * scaling_factor)
    fig.text(0.01, 0.5, 'Distance per Truck / km', ha='center', va='center', rotation='vertical', fontsize=16 * scaling_factor)
    
    # Define legend handle
    mean_line = mlines.Line2D([], [], color=colors['TUMGreen3'], linestyle='--', label='Mean', linewidth=3 * scaling_factor)
    median_line = mlines.Line2D([], [], color=colors['TUMOrange'], label='Median', linewidth=3 * scaling_factor)
    fig.legend(handles=[median_line, mean_line], loc='upper center', fontsize=14 * scaling_factor, frameon=True, ncol=2)

    # Remove any empty subplots if there are fewer than 6 freight forwarders
    for i in range(len(freight_forwarders), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout(rect=[0.02, 0.03, 1, 0.95])  # Adjust layout to make room for common labels
    # plt.savefig('output/figures/operational/freight_forwarder_weekday_boxplots.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/freight_forwarder_weekday_boxplots.pdf', bbox_inches='tight')
   
plot_weekly_distance_boxplot(df_trips)

def verify_weekday_aggregation(df_trips):
    print(plt.style)
    df_trips['start_time'] = pd.to_datetime(df_trips['start_time'], utc=True)
    df_trips['weekday'] = df_trips['start_time'].dt.weekday  # 0 = Montag, 6 = Sonntag
    df_trips['date'] = df_trips['start_time'].dt.date
    daily_distance = df_trips.groupby(['vehicle_id', 'date', 'weekday'])['distance_km'].sum().reset_index()

    print("Verifying weekday aggregation:")
    for weekday in range(7):  # Überprüfen aller Wochentage
        print(f"Sample data for {'Monday' if weekday == 0 else 'Tuesday' if weekday == 1 else 'Wednesday' if weekday == 2 else 'Thursday' if weekday == 3 else 'Friday' if weekday == 4 else 'Saturday' if weekday == 5 else 'Sunday'}:")
        sample_data = daily_distance[daily_distance['weekday'] == weekday].sample(5, random_state=1)  # Zeigt 5 zufällige Stichproben
        print(sample_data)


verify_weekday_aggregation(df_trips)

### Tour Duration and Distance  


In [ ]:

# -------------------------------------------------------------------------------------------
#                              TOUR DURATION & DISTANCE HISTOGRAMS
# -------------------------------------------------------------------------------------------


def plot_tour_duration_distance_histogram(df_trips, max_distance=800, max_duration=30):
    """
    Plot histograms for cumulative tour durations and distances with lines per freight forwarder.
    Tours are shown as percentage of total tours for each freight forwarder.
    
    Parameters:
    -----------
    df_trips : pandas DataFrame
        DataFrame containing trip data with tour_id, freight_forwarder, distance_km, and duration_h columns
    max_distance : float, optional
        Maximum distance to display in km (only affects plot display, not statistics calculation)
    max_duration : float, optional
        Maximum duration to display in hours (only affects plot display, not statistics calculation)
    """
    # First, calculate cumulative values by tour_id
    tour_aggregates = df_trips.groupby(['tour_id', 'freight_forwarder', 'vehicle_id']).agg({
        'distance_km': 'sum',  # Sum of all distances in the tour
        'duration_h': 'sum'    # Sum of all durations in the tour
    }).reset_index()
    
    textwidth = 159.2 / 25.4  # Convert text width from mm to inches
    h_169 = 9 / 16 * textwidth
    width = textwidth * 1.2
    
    # Create figure with GridSpec for better control over subplot placement
    fig = plt.figure(figsize=(width, h_169))
    gs = plt.GridSpec(2, 1, height_ratios=[0.1, 0.9])
    
    # Create area for legend at the top
    legend_ax = fig.add_subplot(gs[0])
    legend_ax.axis('off')  # Hide axes for the legend area
    
    # Create main plotting area with two side-by-side subplots
    plot_gs = gs[1].subgridspec(1, 2, wspace=0.25)
    ax1 = fig.add_subplot(plot_gs[0])
    ax2 = fig.add_subplot(plot_gs[1])
    axes = [ax1, ax2]
    
    # Create a color mapping for freight forwarders
    freight_forwarders = sorted(tour_aggregates['freight_forwarder'].unique())
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]
    
    # Create palette that assigns colors based on freight forwarder
    palette = {vehicle_id: ff_colors[ff] for vehicle_id, ff in 
              tour_aggregates[['vehicle_id', 'freight_forwarder']].drop_duplicates().values}
    
    # Create visualization data (filtered for display purposes)
    duration_data_viz = tour_aggregates[tour_aggregates['duration_h'] <= max_duration]
    distance_data_viz = tour_aggregates[tour_aggregates['distance_km'] <= max_distance]
    
    # Keep track of handles for the legend
    handles = []
    labels = []
    
    # Calculate total number of tours per freight forwarder for percentage calculation
    # Use the COMPLETE dataset for statistics
    ff_tour_counts = tour_aggregates.groupby('freight_forwarder').size().to_dict()
    
    # Create histograms for each freight forwarder
    for ff in freight_forwarders:
        # Get total number of tours for this freight forwarder
        total_ff_tours = ff_tour_counts[ff]
        
        # Duration histogram - use filtered data for visualization only
        ff_duration_data_viz = duration_data_viz[duration_data_viz['freight_forwarder'] == ff]
        # Calculate weights for percentage (each tour will count as 1/total tours * 100%)
        duration_weights = np.ones(len(ff_duration_data_viz)) / total_ff_tours * 100
        
        sns.histplot(
            data=ff_duration_data_viz, 
            x='duration_h',
            bins=40,
            element="step", 
            fill=False,
            color=ff_colors[ff],
            ax=axes[0],
            label=f'FF {ff}',
            weights=duration_weights  # Use weights for percentage
        )
        
        # Distance histogram - use filtered data for visualization only
        ff_distance_data_viz = distance_data_viz[distance_data_viz['freight_forwarder'] == ff]
        # Calculate weights for percentage
        distance_weights = np.ones(len(ff_distance_data_viz)) / total_ff_tours * 100
        
        sns.histplot(
            data=ff_distance_data_viz, 
            x='distance_km',
            bins=40,
            element="step", 
            fill=False,
            color=ff_colors[ff],
            ax=axes[1],
            weights=distance_weights  # Use weights for percentage
        )
        
        # Add handle for legend (only need to do this once per freight forwarder)
        handles.append(plt.Line2D([0], [0], color=ff_colors[ff], lw=2))
        labels.append(f'FF {ff}')
    
    # Set titles and labels
    axes[0].set_title('Tour Duration Distribution')
    axes[1].set_title('Tour Distance Distribution')
    
    # Configure tick locations
    axes[0].xaxis.set_major_locator(ticker.MultipleLocator(5))
    axes[1].xaxis.set_major_locator(ticker.MultipleLocator(100))
    axes[0].xaxis.set_minor_locator(ticker.MultipleLocator(1))
    axes[1].xaxis.set_minor_locator(ticker.MultipleLocator(25))
    axes[0].yaxis.set_minor_locator(ticker.MultipleLocator(2))
    axes[1].yaxis.set_minor_locator(ticker.MultipleLocator(2))
    
    # Add grid lines
    axes[0].grid(which='major', alpha=alpha_major)
    axes[1].grid(which='major', alpha=alpha_major)
    
    # Set axis limits and labels
    axes[0].set(xlim=(0, max_duration), xlabel='Tour Duration / h', ylabel='% of Tours')
    axes[1].set(xlim=(0, max_distance), xlabel='Tour Distance / km', ylabel='')
    
    # Format y-axis labels as percentages
    axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))
    axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))
    
    # Remove individual legends from subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add a single legend at the top of the figure
    legend_ax.legend(handles=handles, labels=labels, ncol=len(handles), loc='center', 
                     title='Freight Forwarder', fontsize=8, title_fontsize=8)
    
    # Set consistent font size
    setFontSize(axes, 9)
    
    # Add statistics to the plot - calculated from the COMPLETE dataset
    for i, (label, ax) in enumerate([('duration_h', axes[0]), ('distance_km', axes[1])]):
        # Use the original, unfiltered data for statistics
        full_data = tour_aggregates[label]
        mean_val = full_data.mean()
        median_val = full_data.median()
        unit = 'h' if i == 0 else 'km'
        
        # Add text with statistics in the upper right corner
        ax.text(0.95, 0.95, f'Mean: {mean_val:.1f} {unit}\nMedian: {median_val:.1f} {unit}', 
                transform=ax.transAxes, ha='right', va='top',
                fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    
    # plt.savefig('output/figures/operational/tour_duration_and_distance_hist.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/tour_duration_and_distance_hist.pdf', bbox_inches='tight')

plot_tour_duration_distance_histogram(df_trips, max_distance=900, max_duration=35)


In [ ]:

# -------------------------------------------------------------------------------------------
#                                Tour DURATION & DISTANCE ECDF
# -------------------------------------------------------------------------------------------


def plot_tour_duration_distance_ecdf(df_trips, max_distance=800, max_duration=30):
    """
    Plot empirical cumulative distribution functions (ECDF) for cumulative tour durations and distances.
    Shows the percentage of tours below certain thresholds for each freight forwarder.
    
    Parameters:
    -----------
    df_trips : pandas DataFrame
        DataFrame containing trip data with tour_id, freight_forwarder, distance_km, and duration_h columns
    max_distance : float, optional
        Maximum distance to display in km (only affects plot display, not statistics calculation)
    max_duration : float, optional
        Maximum duration to display in hours (only affects plot display, not statistics calculation)
    """
    # First, calculate cumulative values by tour_id
    tour_aggregates = df_trips.groupby(['tour_id', 'freight_forwarder', 'vehicle_id']).agg({
        'distance_km': 'sum',  # Sum of all distances in the tour
        'duration_h': 'sum'    # Sum of all durations in the tour
    }).reset_index()
    
    textwidth = 159.2 / 25.4  # Convert text width from mm to inches
    h_169 = 9 / 16 * textwidth
    width = textwidth * 1.2
    
    # Create figure with GridSpec for better control over subplot placement
    fig = plt.figure(figsize=(width, h_169))
    gs = plt.GridSpec(2, 1, height_ratios=[0.1, 0.9])
    
    # Create area for legend at the top
    legend_ax = fig.add_subplot(gs[0])
    legend_ax.axis('off')  # Hide axes for the legend area
    
    # Create main plotting area with two side-by-side subplots
    plot_gs = gs[1].subgridspec(1, 2, wspace=0.25)
    ax1 = fig.add_subplot(plot_gs[0])
    ax2 = fig.add_subplot(plot_gs[1])
    
    # Create a color mapping for freight forwarders
    freight_forwarders = sorted(tour_aggregates['freight_forwarder'].unique())
    ff_colors = {}
    for i, ff in enumerate(freight_forwarders):
        ff_colors[ff] = colors_plot[i % len(colors_plot)]
    
    # Keep track of handles for the legend
    handles = []
    labels = []
    
    # Create ECDF plots for each freight forwarder
    for ff in freight_forwarders:
        ff_data = tour_aggregates[tour_aggregates['freight_forwarder'] == ff]
        
        # Duration ECDF
        sns.ecdfplot(
            data=ff_data, 
            x='duration_h',
            color=ff_colors[ff],
            ax=ax1,
            linewidth=2,
            label=f'FF {ff}'
        )
        
        # Distance ECDF
        sns.ecdfplot(
            data=ff_data, 
            x='distance_km',
            color=ff_colors[ff],
            ax=ax2,
            linewidth=2
        )
        
        # Add handle for legend (only need to do this once per freight forwarder)
        handles.append(plt.Line2D([0], [0], color=ff_colors[ff], lw=2))
        labels.append(f'FF {ff}')
    
    # Set titles and labels
    ax1.set_title('Tour Duration')
    ax2.set_title('Tour Distance')
    
    # Configure tick locations
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax2.xaxis.set_major_locator(ticker.MultipleLocator(200))
    ax1.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax2.xaxis.set_minor_locator(ticker.MultipleLocator(25))
    
    # Add grid lines
    ax1.grid(which='major', alpha=alpha_major)
    ax2.grid(which='major', alpha=alpha_major)
    ax1.grid(which='minor', alpha=alpha_minor)
    ax2.grid(which='minor', alpha=alpha_minor)
    
    # Set axis limits and labels with max limits
    ax1.set(xlim=(0, max_duration), xlabel='Tour Duration / h', ylabel='Cumulative Probability')
    ax2.set(xlim=(0, max_distance), xlabel='Tour Distance / km', ylabel='')
    
    # Format y-axis as percentage
    ax1.yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals=0))
    ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1.0, decimals=0))
    
    # Remove individual legends from subplots
    for ax in [ax1, ax2]:
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    
    # Add reference lines at key percentiles (25%, 50%, 75%) - calculated from COMPLETE dataset
    percentiles = [0.25, 0.5, 0.75]
    linestyles = [':', '--', '-.']
    
    for p, ls in zip(percentiles, linestyles):
        ax1.axhline(y=p, linestyle=ls, color='gray', alpha=0.7, linewidth=1)
        ax2.axhline(y=p, linestyle=ls, color='gray', alpha=0.7, linewidth=1)
        
        # Add percentile values for the complete dataset (not filtered by max limits)
        duration_percentile = tour_aggregates['duration_h'].quantile(p)
        distance_percentile = tour_aggregates['distance_km'].quantile(p)
        
        # Only show vertical lines if they fall within the display range
        if duration_percentile <= max_duration:
            ax1.axvline(x=duration_percentile, linestyle=ls, color='gray', alpha=0.7, linewidth=1)
        if distance_percentile <= max_distance:
            ax2.axvline(x=distance_percentile, linestyle=ls, color='gray', alpha=0.7, linewidth=1)
    
    # Add a single legend at the top of the figure
    legend_ax.legend(handles=handles, labels=labels, ncol=len(handles), loc='center', 
                     title='Freight Forwarder', fontsize=8, title_fontsize=8)
    
    # Set consistent font size
    setFontSize([ax1, ax2], 9)
    
    # Add statistics annotations - calculated from COMPLETE dataset
    for i, (label, ax) in enumerate([('duration_h', ax1), ('distance_km', ax2)]):
        full_data = tour_aggregates[label]
        median_val = full_data.median()
        q75_val = full_data.quantile(0.75)
        unit = 'h' if i == 0 else 'km'
        
        # Add text with key percentiles in the lower right corner
        ax.text(0.95, 0.05, f'Median: {median_val:.1f} {unit}\n75th percentile: {q75_val:.1f} {unit}', 
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=9, bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round'))
    
    # plt.savefig('output/figures/operational/tour_duration_and_distance_ecdf.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/tour_duration_and_distance_ecdf.pdf', bbox_inches='tight')

plot_tour_duration_distance_ecdf(df_trips, max_distance=1300, max_duration=35)

In [ ]:

# ------------------------------------------------------------------------------
#                              REST TIME KDEs BY LOCATION
# ------------------------------------------------------------------------------


def plot_rest_time_kde(df_stops):
    """
    Plot KDEs for rest times segmented by location.
    Uses the same colors for locations as in plot_fleet_occupation() for consistency.
    """
    # Define exact color mapping to match the screenshot legend
    location_colors = {
        'industrial area': colors_plot[2] ,   # blue
        'rest area': colors_plot[4],         # purple/pink
        'home base': colors_plot[1],         # orange
        'other area': colors_plot[3],        # green
        'service area fuel': colors_plot[5]  # yellow
    }

    textwidth = 159.2 / 25.4  # Convert text width from mm to inches
    h_43 = 6  # Define plot height
    cut_time_1 = 2  # First cut-off time for rest in hours
    cut_time_2 = 6  # Second cut-off time for rest in hours
    cut_time_max = 20  # Maximum time for rest in hours
    indicator_level_main = 2.5  # Main indicator level for highlighting
    indicator_level_sub_1 = 2.5  # Sub-indicator level for short rests
    indicator_level_sub_2 = 0.4  # Sub-indicator level for long rests
    indicator_color = '#202020'  # Color for indicator lines

    gf_loc = GridFigure(1, 1, textwidth, h_43 / 2, wspace=0.2, hspace=0.3, legend_mode='above', constrained_layout=False)
    gf_loc_2 = GridFigure(1, 2, textwidth, h_43 / 2, wspace=0.2, constrained_layout=False)
    gf_loc_axes = np.atleast_1d(gf_loc.axes_list).flatten()
    gf_loc_2_axes = np.atleast_1d(gf_loc_2.axes_list).flatten()

    plot_data = df_stops.loc[(df_stops.rest_time <= 24 * 3600)]
    plot_data_2h = df_stops.loc[(df_stops.rest_time <= cut_time_1 * 3600)]
    plot_data_24h = df_stops.loc[(df_stops.rest_time > cut_time_2 * 3600) & (df_stops.rest_time <= 24 * 3600)]

    for data in [plot_data, plot_data_2h, plot_data_24h]:
        if 'rest_time_h' not in data.columns or 'location' not in data.columns:
            raise ValueError("Missing required columns in the data")
        if data[['rest_time_h', 'location']].isnull().any().any():
            raise ValueError("NaNs found in required columns")

    for data, bw_adjust, ax in zip([plot_data, plot_data_2h, plot_data_24h], [.002, .2, .2], [gf_loc_axes[0], *gf_loc_2_axes]):
        # Use the exact color palette from the screenshot
        sns.kdeplot(data=data, x='rest_time_h', hue='location', bw_adjust=bw_adjust, palette=location_colors, cut=0, common_norm=False, ax=ax)

    gf_loc_axes[0].plot([0, cut_time_1], [indicator_level_main] * 2, "--v", linewidth=1, markersize=5, color=indicator_color)
    gf_loc_axes[0].plot([cut_time_2, cut_time_max], [indicator_level_main] * 2, "--o", linewidth=1, markersize=5, color=indicator_color)
    gf_loc_2_axes[0].plot([0, cut_time_1], [indicator_level_sub_1] * 2, "--v", linewidth=1, markersize=5, color=indicator_color)
    gf_loc_2_axes[1].plot([cut_time_2, cut_time_max], [indicator_level_sub_2] * 2, "--o", linewidth=1, markersize=5, color=indicator_color)

    gf_loc.legend_ax.axis("off")

    for ax, major_x_grid, minor_x_grid, major_y_grid, minor_y_grid, xlim, ylim, xlabel, ylabel in zip(
            [gf_loc_axes[0], *gf_loc_2_axes], [4, 0.5, 2], [0.5, 0.1, 1], [0.5, 0.5, 0.1], [0.25, 0.1, 0.05],
            [(0, 24), (0, cut_time_1), (cut_time_2, cut_time_max)], [(0, 3), (0, 3), (0, 0.5)], ['rest time / h'] * 3,
            ['density'] * 2 + ['']):
        ax.grid(which='major', alpha=alpha_major)
        ax.grid(which='minor', alpha=alpha_minor)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(major_x_grid))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(minor_x_grid))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(major_y_grid))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(minor_y_grid))
        ax.set(xlim=xlim, ylim=ylim, xlabel=xlabel, ylabel=ylabel)
        ax.get_legend().remove()
        setFontSize([ax], 9)

    # Create legend with the consistent colors from screenshot
    setFontSize(gf_loc_axes, 9)
    handles = [genLineLegendHandle(location_colors[loc]) for loc in location_colors]
    legend_labels = list(location_colors.keys())
    gf_loc.legend_ax.legend(handles=handles, labels=legend_labels, ncol=3, loc="center")

    # gf_loc.fig.savefig(f'output/figures/operational/rest_time_global.svg', bbox_inches='tight')
    # gf_loc_2.fig.savefig(f'output/figures/operational/rest_time_detail.svg', bbox_inches='tight')

    # gf_loc.fig.savefig(f'output/figures/operational/rest_time_global.pdf', bbox_inches='tight')
    # gf_loc_2.fig.savefig(f'output/figures/operational/rest_time_detail.pdf', bbox_inches='tight')

plot_rest_time_kde(df_stops)

## Data Quality
Gaps in recording in relation to recorded distance

In [ ]:
total_signal_loss = df_trips['n_signal_loss'].sum()
avg_signal_loss_ratio = df_trips['r_signal_loss'].mean()

print(f'Total signal loss: {total_signal_loss}')
print(f'Average signal loss ratio: {avg_signal_loss_ratio}')

## Fleet occupation 

In [ ]:


# ------------------------------------------------------------------------------
#                             FLEET OCCUPATION PLOT
# ------------------------------------------------------------------------------


def plot_fleet_occupation(df_occ, truck_day):

    # Prepare figure
    textwidth = 159.2 / 25.4  # Convert text width from mm to inches
    h_169 = 9 / 16 * textwidth
    gridfig = GridFigure(1,2,textwidth, h_169, wspace=0.35, hspace=.5, legend_mode="above", constrained_layout=False, ratios={'height':[1], 'width':[1, 2]})
    ax = gridfig.axes_list[0] #len(df_occ['freight_forwarder'].unique())]
    LOCATIONS = dp.LOCATIONS
    
    #colors_fleets = {i: color_list[i - 1] for i in range(1, 5)}
    #colors_locations = {loc: color_list[i + 4] for i, loc in enumerate([*LOCATIONS, 'other_area', 'driving'])}
    #colors_locations_2 = {loc.replace('_', ' '): color_list[i + 4] for i, loc in enumerate([*LOCATIONS, 'other_area', 'driving'])}

    # Plot
    # df_occ: absolute numbers per vehicle in each occupation 
    plot_data = df_occ.drop(columns=['freight_forwarder']).transpose()
    plot_data = (plot_data / plot_data.sum()).transpose() #relative values per vehicle
    plot_data['freight_forwarder']=df_occ['freight_forwarder']
    plot_data = plot_data.fillna(0)
    plot_data = plot_data.groupby('freight_forwarder').agg({
        'driving' : 'mean',
        'home_base' : 'mean',
        'industrial_area' : 'mean',
        'other_area' : 'mean',
        'rest_area' : 'mean',
        'service_area_fuel' : 'mean'
    })
    #plot_data = plot_data[['driving','home_base','industrial_area','other_area','rest_area','service_area_fuel']] #rearange columns
    plot_data.plot.bar(ax = ax, stacked=True, color=colors_plot, width=0.5)

    """
    for fleet in plot_data['freight_forwarder'].unique():
        plot_data_fleet = plot_data[plot_data['freight_forwarder']==fleet]
        plot_data_fleet = plot_data_fleet[['driving','home_base','industrial_area','other_area','rest_area','service_area_fuel']] #rearange columns
        plot_data_fleet = plot_data_fleet.mean().to_frame().T
        plot_data_fleet.plot.bar(ax = ax[fleet - 1], stacked=True, color=colors_fleets, width=0.5)
    """

    #plot_data.plot.bar(ax = ax, stacked=True, color=colors_locations, width=0.5)

    # Adjust figure
    ax.set(
        ylim=(0,1),
        yticks=np.arange(0,1.04,0.1),
        xlabel="fleet",
        ylabel="share of vehicle status by fleet"
    )

    ax.grid(axis="y", which='major', alpha=alpha_major)
    ax.grid(axis="y", which='minor', alpha=alpha_minor)

    # Create legend - Apply percentage formatter more explicitly
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(.05))

    handles,labels = ax.get_legend_handles_labels()
    gridfig.legend_ax.axis("off")
    gridfig.legend_ax.legend(handles = handles, labels=[l.replace("_"," ") for l in labels], ncol=3, loc="center")
    ax.get_legend().remove()

    # Adjust fonts
    ax = gridfig.axes_list[1]

    """ -------- 2nd plot -------- """
    # Plot
    plot_data = (truck_day / truck_day.sum()).transpose()
    plot_data.plot.bar(ax=ax, stacked=True, color=colors_plot)

    # Adjust figure
    ax.set(
        ylim=(0,1),
        yticks=np.arange(0,1.04,0.1),
        xlabel="hour",
        ylabel="share of vehicle status by time"
    )

    # Create legend - Apply percentage formatter more explicitly
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0, decimals=0))
    #ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda t, pos: f"{round(t*100)}%"))
    handles,labels = ax.get_legend_handles_labels()
    gridfig.legend_ax.axis("off")
    gridfig.legend_ax.legend(handles = handles, labels=[l.replace("_"," ") for l in labels], ncol=3, loc="center")
    ax.get_legend().remove()

    ax.yaxis.set_minor_locator(ticker.MultipleLocator(.05))

    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(2))

    ax.grid(axis="y", which='major', alpha=alpha_major)
    ax.grid(axis="y", which='minor', alpha=alpha_minor)

    for ax_i in gridfig.axes_list:
        for tick in ax_i.get_xticklabels():
            tick.set(rotation=0)#,ha='right')

    # Adjust fonts
    setFontSize(gridfig.axes_list+[gridfig.legend_ax],9)

    # Save figure
    # plt.savefig('output/figures/operational/fleet_occupation.svg', bbox_inches='tight')
    # plt.savefig('output/figures/operational/fleet_occupation.pdf', bbox_inches='tight')


plot_fleet_occupation(df_rt_joined_plot, truck_day_occ)

### Average departure and arrival times

In [ ]:
def avg_departure_and_arrival(): 

    df = pd.read_csv('input/stations/tours.csv')

    def circular_mean_hour(hours):
        """Calculate circular mean of hours (0-24)"""
        # Convert hours to radians (0-24 hours -> 0-2π)
        radians = hours * (2 * np.pi / 24)
        
        # Calculate mean of x and y components
        x = np.cos(radians).mean()
        y = np.sin(radians).mean()
        
        # Convert back to hours
        mean_radians = np.arctan2(y, x)
        mean_hour = (mean_radians * 24 / (2 * np.pi)) % 24
        
        return mean_hour



    def calculate_average_times(df, start_col='start_time', stop_col='stop_time'):
        """
        Calculate average start and stop times (hour of day) preserving local timezone
        and handling the circular nature of time.
        
        Parameters:
        -----------
        df : pandas.DataFrame
            DataFrame containing start and stop time columns
        start_col : str
            Name of the start time column
        stop_col : str
            Name of the stop time column
            
        Returns:
        --------
        tuple
            (avg_start_hour, avg_stop_hour) as float values in 24h format
        """
        # Make a copy to avoid modifying the original dataframe
        df_copy = df.copy()
        
        # Ensure timestamps are datetime objects (with timezone if present)
        for col in [start_col, stop_col]:
            if not pd.api.types.is_datetime64_any_dtype(df_copy[col]):
                df_copy[col] = pd.to_datetime(df_copy[col], format='mixed')
        
        # Extract hour and minute components (preserving timezone)
        start_hours = df_copy[start_col].apply(lambda x: x.hour + x.minute/60)
        stop_hours = df_copy[stop_col].apply(lambda x: x.hour + x.minute/60)
        
        # Use circular mean for proper time averaging
        avg_start_hour = circular_mean_hour(start_hours)
        avg_stop_hour = circular_mean_hour(stop_hours)
        # Convert average hours to hours and minutes
        avg_start_hour_h = int(avg_start_hour)
        avg_start_hour_min = int((avg_start_hour - avg_start_hour_h) * 60)
        avg_stop_hour_h = int(avg_stop_hour)
        avg_stop_hour_min = int((avg_stop_hour - avg_stop_hour_h) * 60)
        
        print(f"Average start time: {avg_start_hour_h}h {avg_start_hour_min}min")
        print(f"Average stop time: {avg_stop_hour_h}h {avg_stop_hour_min}min \n")
        return avg_start_hour, avg_stop_hour


    freight_forwarder_groups = df.groupby('freight_forwarder')

    for ff, group in freight_forwarder_groups:
        print(f"Freight Forwarder: {ff}")
        avg_start, avg_stop = calculate_average_times(group)


avg_departure_and_arrival()